# Roofline analysis for LLM serving

Builds a roofline model from scratch and uses it to classify each stage of
LLM serving (prefill, decode, KV reload) as compute-bound or memory-bound.
No GPU required: this notebook is about analytical modelling. The measured
TFLOPs and GB/s numbers come from the devices listed in a small datasheet
dictionary so the roofline is representative without needing a GPU on hand.

**Learning objectives**
- Compute arithmetic intensity for matmul, attention, and KV-cache reads.
- Intersect a memory-bandwidth line with a compute peak to locate the ridge.
- Classify prefill, decode, and a multi-query attention kernel by intensity.

**Papers**: Williams et al. *Roofline: An Insightful Visual Performance Model*, CACM 2009.
**Hardware**: CPU. Uses published per-device peaks; no actual kernel is measured.
**Estimated runtime**: ≈30 s.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "scoring" / "harness.py").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "src"))

import numpy as np

from llm_infra_lab._utils import set_seed
from scoring.harness import Scorer

set_seed(0)
s = Scorer("05_serving_01_roofline_analysis")


In [ ]:
# (peak memory bandwidth GB/s, peak FP16/BF16 TFLOPs)
DEVICES: dict[str, dict[str, float]] = {
    "T4":   {"bw": 320.0,  "compute": 65.0,  "capability": 7.5},
    "L4":   {"bw": 300.0,  "compute": 120.0, "capability": 8.9},
    "A10G": {"bw": 600.0,  "compute": 125.0, "capability": 8.6},
    "A100": {"bw": 2039.0, "compute": 312.0, "capability": 8.0},
    "H100": {"bw": 3350.0, "compute": 756.0, "capability": 9.0},
}


def ridge_intensity(bw_gbps: float, compute_tflops: float) -> float:
    return compute_tflops * 1e12 / (bw_gbps * 1e9)


for name, d in DEVICES.items():
    d["ridge"] = ridge_intensity(d["bw"], d["compute"])
    print(f"{name:>5} ridge: {d['ridge']:6.1f} FLOPs/byte")


In [ ]:
def matmul_intensity(m: int, k: int, n: int, dtype_bytes: int = 2) -> float:
    '''Arithmetic intensity of a dense matmul reading both inputs from HBM.'''
    flops = 2 * m * k * n
    bytes_ = dtype_bytes * (m * k + k * n + m * n)
    return flops / bytes_


def attention_intensity(
    batch: int, n_heads: int, head_dim: int, seq_q: int, seq_k: int, dtype_bytes: int = 2
) -> float:
    '''AI for standard SDPA: 2 matmuls + softmax + output matmul; we count the two matmuls.'''
    # Q K^T + softmax(...) V : FLOPs ≈ 2*B*H*Tq*Tk*D + 2*B*H*Tq*Tk*D
    flops = 4 * batch * n_heads * seq_q * seq_k * head_dim
    # Reads: Q (B H Tq D), K (B H Tk D), V (B H Tk D); writes O (B H Tq D).
    bytes_ = dtype_bytes * batch * n_heads * head_dim * (seq_q + 2 * seq_k + seq_q)
    return flops / bytes_


def decode_step_intensity(
    batch: int,
    params: int,
    n_kv_heads: int,
    head_dim: int,
    seq_k: int,
    dtype_bytes: int = 2,
) -> float:
    '''Full decode-step intensity: weight matmuls + attention KV reads.

    Weights are read once per step and shared across the batch (hence the
    :math:`2 \cdot P` byte term); attention reads the per-request KV cache
    (hence the per-request scaling of the attention byte term). Batching
    therefore amortises the weight read and pushes intensity up toward the ridge.
    '''
    weight_flops = 2 * params * batch
    weight_bytes = dtype_bytes * params
    attn_flops = 4 * batch * n_kv_heads * seq_k * head_dim
    attn_bytes = dtype_bytes * batch * n_kv_heads * head_dim * (1 + 2 * seq_k + 1)
    return (weight_flops + attn_flops) / (weight_bytes + attn_bytes)


# 7B-class model with GQA (n_kv_heads=8, head_dim=128). Parameter count
# is ~7 billion; using 7e9 directly for the intensity model.
PARAMS_7B = 7_000_000_000

WORKLOADS: dict[str, float] = {
    "prefill_B1_T1024_matmul_7B": matmul_intensity(1024, 4096, 11008),
    "prefill_B8_T1024_matmul_7B": matmul_intensity(8 * 1024, 4096, 11008),
    "decode_B1_T1_kv2048":         decode_step_intensity(
        batch=1, params=PARAMS_7B, n_kv_heads=8, head_dim=128, seq_k=2048
    ),
    "decode_B32_T1_kv2048":        decode_step_intensity(
        batch=32, params=PARAMS_7B, n_kv_heads=8, head_dim=128, seq_k=2048
    ),
    "flash_attn_B1_H32_T2048":     attention_intensity(batch=1, n_heads=32, head_dim=128, seq_q=2048, seq_k=2048),
}

for k, v in WORKLOADS.items():
    print(f"{k:<35} {v:7.1f} FLOPs/byte")


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
x = np.logspace(-1, 3.5, 400)
for name, d in DEVICES.items():
    y = np.minimum(d["bw"] * 1e9 * x / 1e12, np.full_like(x, d["compute"]))
    ax.loglog(x, y, label=f"{name} (ridge@{d['ridge']:.0f})")

# Plot the workloads as vertical lines.
for w_name, ai in WORKLOADS.items():
    ax.axvline(ai, linestyle=":", alpha=0.4, color="black")
    ax.annotate(w_name, xy=(ai, 1), xytext=(ai * 1.02, 1 + 2 * list(WORKLOADS).index(w_name)),
                fontsize=7, rotation=60, ha="left", va="bottom")

ax.set_xlabel("arithmetic intensity (FLOPs / byte)")
ax.set_ylabel("throughput (TFLOPs)")
ax.set_title("Roofline: LLM serving workloads across common inference GPUs")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Prefill with a large inner-dim matmul at batch ≥1 should be compute-bound
# on every device we care about (intensity above 100 FLOPs/byte).
s.check(
    "prefill_compute_bound_on_T4",
    lambda: WORKLOADS["prefill_B1_T1024_matmul_7B"] > DEVICES["T4"]["ridge"],
    msg=f"prefill AI={WORKLOADS['prefill_B1_T1024_matmul_7B']:.1f} T4 ridge={DEVICES['T4']['ridge']:.1f}",
)
s.check(
    "prefill_compute_bound_on_A100",
    lambda: WORKLOADS["prefill_B8_T1024_matmul_7B"] > DEVICES["A100"]["ridge"],
    msg=f"prefill-batched AI={WORKLOADS['prefill_B8_T1024_matmul_7B']:.1f} A100 ridge={DEVICES['A100']['ridge']:.1f}",
)
# Single-stream decode at batch 1 is firmly memory-bound everywhere (AI ≈ 2).
s.check(
    "decode_bs1_memory_bound_everywhere",
    lambda: all(WORKLOADS["decode_B1_T1_kv2048"] < d["ridge"] for d in DEVICES.values()),
    msg=f"decode AI={WORKLOADS['decode_B1_T1_kv2048']:.1f}",
)
# Batched decode raises AI linearly in batch: B=32 should push decode AI
# within an order of magnitude of the ridge on T4.
s.check(
    "batching_raises_decode_intensity",
    lambda: WORKLOADS["decode_B32_T1_kv2048"] > 2 * WORKLOADS["decode_B1_T1_kv2048"],
    msg=f"B=1 AI={WORKLOADS['decode_B1_T1_kv2048']:.1f} B=32 AI={WORKLOADS['decode_B32_T1_kv2048']:.1f}",
)
# Longer context lowers attention intensity but the B=1 T=2048 attention tile is still above T4 ridge.
s.check(
    "flash_attn_intensity_above_T4_ridge",
    lambda: WORKLOADS["flash_attn_B1_H32_T2048"] > DEVICES["T4"]["ridge"],
    msg=f"FA AI={WORKLOADS['flash_attn_B1_H32_T2048']:.1f} ridge={DEVICES['T4']['ridge']:.1f}",
)


In [ ]:
s.summary()
s.save()
